# Algorithm Forge - Iterative Self-Improvement Loop

**Ziel:** trainiere iterativ immer bessere Mutator-LLMs. V1 lernt von OpenAI-Mutationen, V2 lernt von V1-Mutationen + OpenAI, V3 von V2 + V1 + OpenAI usw.

**Resilient gegen Disconnects:** alles wird in Google Drive persistiert. Wenn die Colab-Session stirbt, einfach das Notebook nochmal **Run all** - es macht da weiter wo es aufgehoert hat.

**Voraussetzungen:**
- Runtime -> T4 GPU
- Colab Secrets: `HF_TOKEN`, `OPENAI_API_KEY`
- Google Drive Zugriff (wird beim ersten Run gefragt)

**Dauer:** ~90 Minuten pro Iteration auf T4. Default = 2 Iterationen = ~3h.

## 1. Konfiguration

In [ ]:
# === HAUPT-KNOEPFE ===
ITERATIONS = 2              # Wie viele V1, V2, V3... Iterationen?
RUNS_PER_BENCHMARK = 4      # Wie viele seeded Runs je Benchmark je Iteration?
GENERATIONS_PER_RUN = 15    # Mutationen pro Run
FT_EPOCHS = 1               # Fine-Tuning Epochen je Iteration

# === Bootstrap-Provider (fuer Iteration 0) ===
BOOTSTRAP_PROVIDER = "openai"
BOOTSTRAP_MODEL = "gpt-4o-mini"   # ~$0.50 fuer eine ganze Iteration

# === Benchmarks ===
BENCHMARKS = ["sort", "matmul"]   # "matmul3" ist zu schwer fuer eine Iter

# === HF Hub ===
HF_USERNAME = "Beko2210"
HF_DATASET_PREFIX = f"{HF_USERNAME}/algorithm-forge-mutations"
HF_MODEL_PREFIX = f"{HF_USERNAME}/algorithm-forge-mutator"

# === Repo ===
REPO_URL = "https://github.com/BEKO2210/Science_game-.git"
BRANCH = "claude/evolution-game-concept-yN3Tn"

# === Persistenz: Google Drive Pfad ===
DRIVE_WORK_DIR = "/content/drive/MyDrive/algorithm-forge"

print(f"Plan: {ITERATIONS} iterations, {RUNS_PER_BENCHMARK} runs x {len(BENCHMARKS)} benchmarks x {GENERATIONS_PER_RUN} gens each")
print(f"Total LLM calls per iteration: {RUNS_PER_BENCHMARK * len(BENCHMARKS) * GENERATIONS_PER_RUN}")
print(f"Estimated time: {ITERATIONS * 90} min")
print(f"Estimated cost: {ITERATIONS} x ~$0.50 = ${ITERATIONS * 0.50:.2f} (for bootstrap iter only; later iters are free)")

## 2. Google Drive mounten (Persistenz!)

In [ ]:
import os, sys, json, subprocess, shutil, time
from pathlib import Path

from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

WORK_DIR = Path(DRIVE_WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "runs").mkdir(exist_ok=True)
(WORK_DIR / "datasets").mkdir(exist_ok=True)
(WORK_DIR / "checkpoints").mkdir(exist_ok=True)
(WORK_DIR / "reports").mkdir(exist_ok=True)
(WORK_DIR / "status").mkdir(exist_ok=True)

print(f"Persistent work dir: {WORK_DIR}")
print("Contents:")
for p in sorted(WORK_DIR.iterdir()):
    print(f"  {p.name}/")

In [ ]:
# Helper: Check ob ein Step schon erledigt ist (Resume-Logik)
def step_done(iteration: int, step: str) -> bool:
    return (WORK_DIR / "status" / f"iter{iteration}_{step}.done").exists()

def mark_step_done(iteration: int, step: str):
    (WORK_DIR / "status" / f"iter{iteration}_{step}.done").write_text(
        f"done at {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

def reset_step(iteration: int, step: str):
    f = WORK_DIR / "status" / f"iter{iteration}_{step}.done"
    if f.exists(): f.unlink()

# Status zeigen
print("=== Aktueller Fortschritt ===")
for status in sorted((WORK_DIR / "status").glob("*.done")):
    print(f"  OK {status.stem}")

## 3. Repo + Dependencies

In [ ]:
REPO_DIR = Path("/content/Science_game-")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--recursive", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

os.chdir(REPO_DIR)

# Bridge runs/datasets/checkpoints aus Drive in das Repo
for sub in ["runs", "datasets", "checkpoints"]:
    local = REPO_DIR / sub
    target = WORK_DIR / sub
    if local.is_symlink() or local.exists():
        if local.is_symlink(): local.unlink()
        elif local.is_dir(): shutil.rmtree(local)
    local.symlink_to(target)
    print(f"  {local} -> {target}")

print("\nCommit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -qU uv
!uv sync --extra dev --extra api-llm 2>&1 | tail -3

In [ ]:
# Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("OpenAI + HF Hub authentifiziert.")

## 4. Unsloth einmalig installieren

In [ ]:
import importlib.util
if importlib.util.find_spec("unsloth") is None:
    !pip install -qU unsloth
    !pip install -qU "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Unsloth ready.")

## 5. Hauptfunktionen

Jeder Schritt ist idempotent - wenn schon `done` markiert, ueberspringt er sich.

In [ ]:
from typing import Optional

def generate_runs(iteration: int, provider: str, model: Optional[str] = None,
                  mutator_obj=None):
    """Step A: generate seeded runs for this iteration."""
    if step_done(iteration, "runs"):
        print(f"[iter {iteration}] runs already generated, skipping.")
        return
    iter_runs_root = Path(f"runs/iter{iteration}")
    iter_runs_root.mkdir(parents=True, exist_ok=True)

    if mutator_obj is None:
        # CLI-Pfad (OpenAI Bootstrap)
        for bench in BENCHMARKS:
            for seed_offset in range(RUNS_PER_BENCHMARK):
                seed = iteration * 1000 + seed_offset
                args = [
                    "uv", "run", "science-game", "run", bench,
                    "--provider", provider,
                    "--generations", str(GENERATIONS_PER_RUN),
                    "--seed", str(seed),
                    "--temperature", "0.9",
                    "--runs-root", str(iter_runs_root),
                ]
                if model: args.extend(["--model", model])
                print(f"  [iter {iteration}] {bench} seed={seed}...", end=" ", flush=True)
                t0 = time.time()
                proc = subprocess.run(args, capture_output=True, text=True)
                if proc.returncode != 0:
                    print(f"FAIL ({proc.stderr[-200:]})")
                    continue
                # extract best fitness from output
                best = [l for l in proc.stdout.splitlines() if "Best fitness" in l]
                print(f"{time.time()-t0:.1f}s  {best[0] if best else ''}")
    else:
        # In-process Pfad (fuer fine-tuned Mutator aus Iter > 0)
        sys.path.insert(0, str(REPO_DIR / "src"))
        from science_game.benchmarks import get_benchmark
        from science_game.engine import Engine, EvolutionConfig
        for bench_name in BENCHMARKS:
            for seed_offset in range(RUNS_PER_BENCHMARK):
                seed = iteration * 1000 + seed_offset
                run_dir = iter_runs_root / f"{bench_name}-seed{seed}"
                cfg = EvolutionConfig(
                    benchmark=bench_name,
                    llm_provider=mutator_obj.name,
                    generations=GENERATIONS_PER_RUN,
                    seed=seed,
                    temperature=0.9,
                    run_dir=run_dir,
                )
                # Manifest fuer Konsistenz mit CLI-Pfad
                from science_game.publish.manifest import Manifest, write_manifest
                from dataclasses import asdict
                mf = Manifest.create(run_dir.name, config=asdict(cfg))
                write_manifest(mf, run_dir / "manifest.json")
                bench = get_benchmark(bench_name)
                print(f"  [iter {iteration}] {bench_name} seed={seed} (fine-tuned mutator)...", end=" ", flush=True)
                t0 = time.time()
                best = Engine(cfg, bench, mutator_obj).run()
                print(f"{time.time()-t0:.1f}s  Best fitness: {best.result.fitness:.6f}")

    mark_step_done(iteration, "runs")

In [ ]:
def aggregate_dataset(iteration: int) -> Path:
    """Step B: aggregate ALL runs from iter 0..N into ONE dataset (cumulative)."""
    out_path = WORK_DIR / "datasets" / f"mutator-v{iteration + 1}.jsonl"
    if step_done(iteration, "dataset") and out_path.exists():
        n = sum(1 for _ in out_path.open()); print(f"[iter {iteration}] dataset already built ({n} examples).")
        return out_path
    # Sammle Daten aus iter 0 bis iter
    cumulative_runs = Path("runs/_cumulative")
    if cumulative_runs.exists(): shutil.rmtree(cumulative_runs)
    cumulative_runs.mkdir(parents=True)
    for i in range(iteration + 1):
        iter_dir = Path(f"runs/iter{i}")
        if not iter_dir.exists(): continue
        for run in iter_dir.iterdir():
            if not run.is_dir(): continue
            dest = cumulative_runs / f"i{i}_{run.name}"
            if not dest.exists():
                shutil.copytree(run, dest)
    print(f"  Cumulative runs (iter 0..{iteration}): {len(list(cumulative_runs.iterdir()))}")
    subprocess.run([
        "uv", "run", "science-game", "build-mutator-dataset",
        "--runs-root", str(cumulative_runs),
        "--out", str(out_path),
        "--mode", "sft",
        "--min-delta", "0.0001",
    ], check=True)
    n = sum(1 for _ in out_path.open())
    print(f"  Dataset: {n} examples -> {out_path}")
    if n < 3:
        raise RuntimeError(f"Dataset too small ({n}). Bootstrap provider quality is too low - try gpt-4o instead of gpt-4o-mini.")
    mark_step_done(iteration, "dataset")
    return out_path

In [ ]:
def push_dataset(iteration: int, dataset_path: Path) -> str:
    """Step C: push dataset to HF Hub."""
    if step_done(iteration, "push_dataset"):
        return f"{HF_DATASET_PREFIX}-v{iteration + 1}"
    from huggingface_hub import HfApi, create_repo
    repo_id = f"{HF_DATASET_PREFIX}-v{iteration + 1}"
    create_repo(repo_id, repo_type="dataset", exist_ok=True, private=False)
    HfApi().upload_file(
        path_or_fileobj=str(dataset_path),
        path_in_repo="data.jsonl",
        repo_id=repo_id, repo_type="dataset",
    )
    print(f"  Dataset hochgeladen: https://huggingface.co/datasets/{repo_id}")
    mark_step_done(iteration, "push_dataset")
    return repo_id

In [ ]:
def fine_tune(iteration: int, dataset_repo: str) -> Path:
    """Step D: fine-tune Qwen2.5-Coder-7B with LoRA on the cumulative dataset.
    Saves an LoRA adapter checkpoint to Drive for later loading."""
    ckpt = WORK_DIR / "checkpoints" / f"mutator-v{iteration + 1}"
    if step_done(iteration, "finetune") and ckpt.exists():
        print(f"[iter {iteration}] fine-tuned checkpoint already exists: {ckpt}")
        return ckpt

    from unsloth import FastLanguageModel
    from datasets import load_dataset
    from unsloth.chat_templates import standardize_sharegpt
    from trl import SFTTrainer
    from transformers import TrainingArguments
    import torch

    # Always start from base Qwen (NOT from previous iter's LoRA) — more data, fresh fit
    print(f"  [iter {iteration}] Loading base Qwen2.5-Coder-7B...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        max_seq_length=4096, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42,
    )

    print(f"  Loading dataset {dataset_repo}...")
    ds = load_dataset(dataset_repo, split="train")
    ds = standardize_sharegpt(ds)
    print(f"  Training on {len(ds)} examples, {FT_EPOCHS} epochs...")

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        dataset_text_field="text", max_seq_length=4096,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, num_train_epochs=FT_EPOCHS, learning_rate=2e-4,
            logging_steps=5, optim="adamw_8bit", seed=42,
            output_dir=f"/content/ft_out_v{iteration + 1}",
            report_to="none",
        ),
    )
    trainer.train()

    ckpt.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ckpt))
    tokenizer.save_pretrained(str(ckpt))
    print(f"  Saved checkpoint -> {ckpt}")

    # Free VRAM before next iteration
    del model, tokenizer, trainer; torch.cuda.empty_cache()

    mark_step_done(iteration, "finetune")
    return ckpt

In [ ]:
def push_model_to_hub(iteration: int, ckpt: Path):
    """Step E: push LoRA adapter to HF Hub."""
    if step_done(iteration, "push_model"):
        return
    from huggingface_hub import HfApi, create_repo
    repo_id = f"{HF_MODEL_PREFIX}-v{iteration + 1}"
    create_repo(repo_id, repo_type="model", exist_ok=True, private=False)
    HfApi().upload_folder(
        folder_path=str(ckpt), repo_id=repo_id, repo_type="model",
        ignore_patterns=["*.bin", "global_step*"],   # nur LoRA adapter + tokenizer
    )
    print(f"  LoRA hochgeladen: https://huggingface.co/{repo_id}")
    mark_step_done(iteration, "push_model")

In [ ]:
def load_finetuned_as_provider(ckpt: Path):
    """Load a fine-tuned LoRA + base model and wrap as an LLMProvider."""
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ckpt),
        max_seq_length=4096, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    sys.path.insert(0, str(REPO_DIR / "src"))
    from science_game.llm.base import LLMProvider, MutationRequest, MutationResponse
    from science_game.llm.ollama_provider import extract_code

    class FineTunedProvider(LLMProvider):
        name = "hf-finetuned"
        def __init__(self, m, t): self.model, self.tokenizer = m, t
        def mutate(self, req):
            prompt = (
                "Below is the current best program. Produce an improved variant. "
                "Return ONLY a single fenced Python code block.\n\n"
                f"```python\n{req.parent_code}\n```\n"
            )
            inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
            with __import__("torch").no_grad():
                out = self.model.generate(
                    **inputs, max_new_tokens=1024,
                    temperature=req.temperature, do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            raw = self.tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            return MutationResponse(
                child_code=extract_code(raw), raw_text=raw,
                provider=self.name, model=str(ckpt.name),
            )
    return FineTunedProvider(model, tokenizer), model, tokenizer

In [ ]:
def ab_compare(iteration: int, ckpt: Path):
    """Step F: A/B compare fine-tuned vs OpenAI bootstrap on fresh seeds."""
    report_path = WORK_DIR / "reports" / f"ab_iter{iteration}.json"
    if step_done(iteration, "ab") and report_path.exists():
        print(f"[iter {iteration}] A/B already done -> {report_path}")
        return json.loads(report_path.read_text())

    sys.path.insert(0, str(REPO_DIR / "src"))
    from science_game.benchmarks import get_benchmark
    from science_game.engine import Engine, EvolutionConfig
    from science_game.llm import get_provider

    ft, model, tokenizer = load_finetuned_as_provider(ckpt)
    base = get_provider(BOOTSTRAP_PROVIDER, model=BOOTSTRAP_MODEL)

    AB_SEEDS = [9000, 9001, 9002]; AB_GEN = 10; AB_BENCH = "sort"
    bench = get_benchmark(AB_BENCH)
    pairs = []
    for s in AB_SEEDS:
        base_dir = Path(f"runs/ab/iter{iteration}/seed{s}-base")
        ft_dir = Path(f"runs/ab/iter{iteration}/seed{s}-ft")
        base_cfg = EvolutionConfig(
            benchmark=AB_BENCH, llm_provider=BOOTSTRAP_PROVIDER, llm_model=BOOTSTRAP_MODEL,
            generations=AB_GEN, seed=s, run_dir=base_dir,
        )
        ft_cfg = EvolutionConfig(
            benchmark=AB_BENCH, llm_provider="hf-finetuned",
            generations=AB_GEN, seed=s, run_dir=ft_dir,
        )
        print(f"  [iter {iteration}] A/B seed {s}...", end=" ", flush=True)
        base_best = Engine(base_cfg, bench, base).run()
        ft_best = Engine(ft_cfg, bench, ft).run()
        delta = ft_best.result.fitness - base_best.result.fitness
        winner = "finetuned" if delta > 1e-9 else ("base" if delta < -1e-9 else "tie")
        print(f"base={base_best.result.fitness:.6f}  ft={ft_best.result.fitness:.6f}  -> {winner}")
        pairs.append({
            "seed": s,
            "base_fitness": base_best.result.fitness,
            "ft_fitness": ft_best.result.fitness,
            "delta": delta, "winner": winner,
        })

    # Cleanup VRAM
    import torch
    del ft, model, tokenizer; torch.cuda.empty_cache()

    report = {"iteration": iteration, "pairs": pairs,
              "avg_delta": sum(p["delta"] for p in pairs) / len(pairs),
              "wins_ft": sum(1 for p in pairs if p["winner"] == "finetuned"),
              "wins_base": sum(1 for p in pairs if p["winner"] == "base"),
              "ties": sum(1 for p in pairs if p["winner"] == "tie")}
    report_path.write_text(json.dumps(report, indent=2))
    print(f"  Report -> {report_path}")
    print(f"  Summary: ft wins {report['wins_ft']}/{len(pairs)}, avg delta {report['avg_delta']:+.6f}")
    mark_step_done(iteration, "ab")
    return report

## 6. Main loop

In [ ]:
all_reports = []
previous_ckpt = None

for iteration in range(ITERATIONS):
    print(f"\n{'=' * 60}\nITERATION {iteration + 1}/{ITERATIONS}\n{'=' * 60}")

    # A) Generate runs
    if iteration == 0:
        generate_runs(iteration, provider=BOOTSTRAP_PROVIDER, model=BOOTSTRAP_MODEL)
    else:
        # Use the LATEST fine-tuned model from previous iter
        print(f"  Loading previous mutator v{iteration} for run generation...")
        ft, model, tok = load_finetuned_as_provider(previous_ckpt)
        generate_runs(iteration, provider="hf-finetuned", mutator_obj=ft)
        import torch
        del ft, model, tok; torch.cuda.empty_cache()

    # B) Aggregate cumulative dataset
    ds_path = aggregate_dataset(iteration)

    # C) Push dataset
    ds_repo = push_dataset(iteration, ds_path)

    # D) Fine-tune
    ckpt = fine_tune(iteration, ds_repo)

    # E) Push model
    push_model_to_hub(iteration, ckpt)

    # F) A/B compare
    report = ab_compare(iteration, ckpt)
    all_reports.append(report)

    previous_ckpt = ckpt

print("\n=== ALL ITERATIONS DONE ===")

## 7. Final-Report & Progress-Plot

In [ ]:
# Lade alle Reports (auch von vorherigen Sessions, durch Drive-Persistenz)
all_report_files = sorted((WORK_DIR / "reports").glob("ab_iter*.json"),
                          key=lambda p: int(p.stem.replace("ab_iter", "")))
all_reports = [json.loads(p.read_text()) for p in all_report_files]

print(f"\n{'iter':<5} {'ft_wins':<10} {'base_wins':<10} {'ties':<6} {'avg_delta':<12}")
print("-" * 50)
for r in all_reports:
    print(f"{r['iteration']:<5} {r['wins_ft']:<10} {r['wins_base']:<10} {r['ties']:<6} {r['avg_delta']:+.6f}")

# Plot
try:
    import matplotlib.pyplot as plt
    iters = [r['iteration'] for r in all_reports]
    deltas = [r['avg_delta'] for r in all_reports]
    plt.figure(figsize=(8, 4))
    plt.plot(iters, deltas, marker='o', linewidth=2)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
    plt.xlabel('Iteration'); plt.ylabel('Avg fitness delta (ft - base)')
    plt.title('Self-improvement progress: fine-tuned mutator vs. OpenAI baseline')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(WORK_DIR / 'reports' / 'progress.png', dpi=100); plt.show()
    print(f"Plot saved -> {WORK_DIR / 'reports' / 'progress.png'}")
except Exception as e:
    print(f"Plot skipped: {e}")

## 8. Verdict

- **avg_delta > 0** und steigend über Iterationen: dein Mutator wird messbar besser. Self-Improvement-Loop bestätigt.
- **avg_delta > 0** aber flach: ein einmaliger Effekt, mehr Iterationen würden nichts bringen.
- **avg_delta ≤ 0**: Bootstrap (GPT-4o-mini) ist auf diesem Benchmark zu stark, der Mutator kann nicht aufschließen. Versuche `BOOTSTRAP_MODEL = "gpt-4o-mini"` bleibt aber mit `BENCHMARKS = ["sort"]` (sort hat mehr Headroom) und `ITERATIONS = 3`.

Alles liegt in `/content/drive/MyDrive/algorithm-forge/` — auch nach Disconnect verfügbar. Beim nächsten Run All überspringt das Notebook alle erledigten Schritte und macht da weiter wo es aufgehört hat.